In [2]:
import pandas as pd
import numpy as np
import pickle

with open('../../cache/ula_v1.pkl', 'rb') as f:
    ula_df = pickle.load(f)

ula_df['book_date'] = pd.to_datetime(ula_df['book_date'])
ula_df['make'] = ula_df['make'].str.upper().str[:3]

for lob_name, lob_filter in [('KMX', 'KMX'), ('STE', 'STG')]:
    subset = ula_df[(ula_df.lob == lob_filter) & (ula_df.book_date >= '2023-01-01')].copy()
    subset['quarter'] = subset['book_date'].dt.to_period('Q')

    subset['hypothetical_car_make_benefit'] = (
        (subset.cd_model_score >= 133) & subset.make.isin({'HON', 'TOY', 'LEX'})
    )

    result = subset.groupby('quarter').agg(
        pct_eligible=('hypothetical_car_make_benefit', 'mean'),
        n_eligible=('hypothetical_car_make_benefit', 'sum'),
        n_total=('hypothetical_car_make_benefit', 'count'),
    ).reset_index()

    result['pct_eligible'] = (result['pct_eligible'] * 100).round(2)
    result['quarter'] = result['quarter'].astype(str)

    print(f'{lob_name} loans that would qualify for car make benefit (HON/TOY/LEX, model score >= 133)\n')
    print(result.to_string(index=False))
    print('\n' + '='*70 + '\n')

KMX loans that would qualify for car make benefit (HON/TOY/LEX, model score >= 133)

quarter  pct_eligible  n_eligible  n_total
 2023Q1         11.69        1681    14375
 2023Q2         12.72        1618    12717
 2023Q3         12.43        1605    12912
 2023Q4         13.43        1417    10549
 2024Q1         12.82        2326    18150
 2024Q2         13.07        1811    13852
 2024Q3         11.85        1543    13020
 2024Q4         11.15        1446    12967
 2025Q1         10.58        2301    21753
 2025Q2         14.24        2628    18451
 2025Q3         16.99        5311    31254
 2025Q4         19.35        9598    49595
 2026Q1         16.14       24219   150070
 2026Q2         17.43       21002   120518
 2026Q3         16.20       16818   103819


STE loans that would qualify for car make benefit (HON/TOY/LEX, model score >= 133)

quarter  pct_eligible  n_eligible  n_total
 2023Q1         15.41         145      941
 2023Q2         16.83         193     1147
 2023Q3    